In [1]:
import torch
import argparse
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd

# Model imports
from Model.Baseline_Models.UNet_plusplus import UnetPlusPlus
from Model.Ablation_Study.DHD_Net_Without_DFA import DHD_Net_Without_DFA as Without_DFA
from Model.Ablation_Study.DHD_Net_Without_DT import DHD_Net_Without_DT as Without_Dual_Task
from Model.DHD_Net.Dual_Task_Fusion_Network import Dual_Task as DHD_Net

# Utils (assuming these exist in your utils.py)
from utils import RSDataset, compute_metrics, run_epoch_DHD_Net

In [2]:
# ------------------------ Arguments ------------------------
parser = argparse.ArgumentParser(description='Evaluate ablation models on test set')
parser.add_argument('--test_img_dir', type=str, default='./Data/Dataset/test/data', help='Test images directory')
parser.add_argument('--test_lab_dir', type=str, default='./Data/Dataset/test/label', help='Test labels directory')
parser.add_argument('--batch_size', type=int, default=8)
parser.add_argument('--class_num', type=int, default=6)
parser.add_argument('--re_size', default=None)
parser.add_argument('--extension_img', type=str, default='tif')
parser.add_argument('--extension_lab', type=str, default='tif')
parser.add_argument('--device', type=str, default='cuda:1')

# Model weight paths
parser.add_argument('--weights_unetpp', type=str, default='./Model_Weight/UNet++.pkl')
parser.add_argument('--weights_without_dt', type=str, default='./Model_Weight/DHD_Net_Without_Dual_Task.pkl')
parser.add_argument('--weights_without_dfa', type=str, default='./Model_Weight/DHD_Net_Without_DFA.pth')
parser.add_argument('--weights_dhd', type=str, default='./Model_Weight/DHD-Net.pkl')

opt = parser.parse_args(args=[])

device = torch.device(opt.device if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# ------------------------ Data Loader ------------------------
test_dataset = RSDataset(
    opt.test_img_dir, opt.test_lab_dir,
    image_size=opt.re_size,
    extension_img=opt.extension_img,
    extension_lab=opt.extension_lab
)
test_loader = DataLoader(test_dataset, batch_size=opt.batch_size, shuffle=False)

criterion = nn.CrossEntropyLoss()  # shared for all models

# ------------------------ Helper evaluation function ------------------------
def evaluate_model(model, model_name):
    model.to(device)
    model.eval()
    with torch.no_grad():
        _, true, pred = run_epoch_DHD_Net(
            test_loader, model, criterion, optimizer=None, is_train=False, device=device
        )
    metrics = compute_metrics(
        true.cpu().numpy(), pred.cpu().numpy(), num_classes=opt.class_num
    )
    # Extract class 1 metrics
    class1 = metrics["Per-Class Metrics"]["Class_1"]
    return {
        "Model": model_name,
        "F1 Score (%)": class1["F1_Score"] * 100,
        "Recall (%)": class1["Recall"] * 100,
        "Precision (%)": class1["Precision"] * 100,
        "IoU (%)": class1["IoU"] * 100
    }

Using device: cuda:1

Found 723 images.


In [3]:
# ------------------------ Model definitions and loading ------------------------
results = []

# 1. UNet++ (baseline)
model_unetpp = UnetPlusPlus(num_classes=opt.class_num, deep_supervision=True)
model_unetpp.load_state_dict(torch.load(opt.weights_unetpp, map_location=device), strict=True)
results.append(evaluate_model(model_unetpp, "UNet++"))

# 2. Without Dual-Task
model_no_dt = Without_Dual_Task(num_classes=opt.class_num)
model_no_dt.load_state_dict(torch.load(opt.weights_without_dt, map_location=device), strict=True)
results.append(evaluate_model(model_no_dt, "Without Dual-Task"))

# 3. Without DFA
model_no_dfa = Without_DFA(num_classes=opt.class_num)
model_no_dfa.load_state_dict(torch.load(opt.weights_without_dfa, map_location=device), strict=True)
results.append(evaluate_model(model_no_dfa, "Without DFA"))

# 4. DHD-Net (full model)
model_dhd = DHD_Net(num_classes=opt.class_num)
model_dhd.load_state_dict(torch.load(opt.weights_dhd, map_location=device), strict=True)
results.append(evaluate_model(model_dhd, "DHD-Net"))

# ------------------------ Print comparison table ------------------------
df = pd.DataFrame(results)
df = df.set_index("Model")
# Reorder columns as requested
df = df[["F1 Score (%)", "Recall (%)", "Precision (%)", "IoU (%)"]]

print("\n" + "="*60)
print("Ablation Study Comparison on Test Set (Class 1 Metrics in %)")
print("="*60)
print(df.round(2))
print("="*60)


Ablation Study Comparison on Test Set (Class 1 Metrics in %)
                   F1 Score (%)  Recall (%)  Precision (%)  IoU (%)
Model                                                              
UNet++                    70.33       70.85          69.82    54.24
Without Dual-Task         75.17       73.23          77.21    60.21
Without DFA               75.25       75.51          75.00    60.32
DHD-Net                   77.55       80.24          75.03    63.33
